# Plow features

This notebook was used to inspect raw plow files and build the derived plow coverage tables. Main derived output used later is `data/derived/plow_df.csv`.


In [ ]:
import requests
import os

dataset_url = "https://data.cityofnewyork.us/resource/rmhc-afj9.csv"
rows_per_chunk = 10_000_000
start_chunk = 0
# Full raw export is paged into 40 chunk files.
for i in range(start_chunk, 40):
    offset = i * rows_per_chunk
    params = {
        "$limit": rows_per_chunk,
        "$offset": offset
    }
    print(f"Downloading chunk {i} with offset {offset}...")
    response = requests.get(dataset_url, params=params)

    if response.status_code != 200:
        print(f"Stopped at chunk {i}, HTTP error {response.status_code}")
        break

    content = response.content
    if len(content) < 500:  # heuristic: empty CSV, just header
        print(f"No more rows at chunk {i}. Done.")
        break

    filename = f"data/raw/plow/plow_{i}.csv"
    with open(filename, "wb") as f:
        f.write(content)

    print(f"Saved {filename}")


In [ ]:
import pandas as pd

years_all = []

for i in {7,15,21}:
    df = pd.read_csv(f"data/raw/plow/plow_{i}.csv", usecols=["snapshot"], on_bad_lines = "skip")
    df["snapshot"] = pd.to_datetime(df["snapshot"], errors="coerce")
    years = df["snapshot"].dt.year.dropna().astype(int)
    years_all.extend(years)

# Convert to a Series and get value counts (how many rows per year)
year_counts = pd.Series(years_all).value_counts().sort_index()

print(year_counts)

In [ ]:
# identifying and dealing with problem in plow_7 and plow_21

import pandas as pd

years_all = []

for i in {7,15,21}:
    df = pd.read_csv(f"data/raw/plow/plow_{i}.csv", usecols=["snapshot"], on_bad_lines = "skip")
    df["snapshot"] = pd.to_datetime(df["snapshot"], errors="coerce")
    years = df["snapshot"].dt.year.dropna().astype(int)
    years_all.extend(years)
    print(i, df.shape[0])

# Convert to a Series and get value counts (how many rows per year)
year_counts = pd.Series(years_all).value_counts().sort_index()

print(year_counts)

In [ ]:
with open("data/raw/plow/plow_21.csv", "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()[-10:]
print("".join(lines))

In [ ]:
with open("data/raw/plow/plow_21.csv", "rb") as f:
    data = f.read().rsplit(b"\n", 1)[0]  # drop last line

with open("data/raw/plow/plow_21.csv", "wb") as f:
    f.write(data)

In [ ]:
# issues have been fixed (7, 15, 21 were not read in properly the first time around. checked now that each one has 10 million rows)

In [ ]:
# now find range of years

import pandas as pd

years_all = []

for i in range(10):
    df = pd.read_csv(f"data/raw/plow/plow_{i}.csv", usecols=["snapshot"], on_bad_lines="skip")
    df["snapshot"] = pd.to_datetime(df["snapshot"], errors="coerce")
    years_all.extend(df["snapshot"].dt.year.dropna().astype(int))

# Count occurrences
year_counts = pd.Series(years_all).value_counts().sort_index().to_dict()

year_counts[2023] = 0

print(year_counts)

In [ ]:
years_next_list = []

for i in range(10, 20):
    df = pd.read_csv(f"data/raw/plow/plow_{i}.csv", usecols=["snapshot"], on_bad_lines="skip")
    df["snapshot"] = pd.to_datetime(df["snapshot"], errors="coerce")
    years = df["snapshot"].dt.year.dropna().astype(int)
    years_next_list.extend(years)

# Convert to Series and count occurrences
year_counts_next = pd.Series(years_next_list).value_counts()

# Merge with previous year_counts dict
for year, count in year_counts_next.items():
    year_counts[year] = year_counts.get(year, 0) + count

# Sort by year
year_counts = dict(sorted(year_counts.items()))

print(year_counts)

In [ ]:
years_next_list = []

for i in range(20, 30):
    df = pd.read_csv(f"data/raw/plow/plow_{i}.csv", usecols=["snapshot"], on_bad_lines="skip")
    df["snapshot"] = pd.to_datetime(df["snapshot"], errors="coerce")
    years = df["snapshot"].dt.year.dropna().astype(int)
    years_next_list.extend(years)

# Convert to Series and count occurrences
year_counts_next = pd.Series(years_next_list).value_counts()

# Merge with previous year_counts dict
for year, count in year_counts_next.items():
    year_counts[year] = year_counts.get(year, 0) + count

# Sort by year
year_counts = dict(sorted(year_counts.items()))

print(year_counts)

In [ ]:
years_next_list = []

for i in range(30, 40):
    df = pd.read_csv(f"data/raw/plow/plow_{i}.csv", usecols=["snapshot"], on_bad_lines="skip")
    df["snapshot"] = pd.to_datetime(df["snapshot"], errors="coerce")
    years = df["snapshot"].dt.year.dropna().astype(int)
    years_next_list.extend(years)

# Convert to Series and count occurrences
year_counts_next = pd.Series(years_next_list).value_counts()

# Merge with previous year_counts dict
for year, count in year_counts_next.items():
    year_counts[year] = year_counts.get(year, 0) + count

# Sort by year
year_counts = dict(sorted(year_counts.items()))

print(year_counts)

In [ ]:
sum(year_counts.values())

check for 2020...

In [ ]:
df = pd.read_csv(f"data/raw/plow/plow_{i}.csv", usecols=["snapshot"], on_bad_lines="skip")

In [ ]:
import pandas as pd

plow_1 = pd.read_csv("data/raw/plow/plow_1.csv")



In [ ]:
plow_1[plow_1["last_visited"] == 57761]

In [ ]:
cscl = pd.read_csv("data/reference/CSCL.csv")

In [ ]:
cscl[cscl["PHYSICALID"] == 8]

In [ ]:
MULTILINESTRING ((-74.22950020729 40.504598328079, -74.229288593263 40.504007445108, -74.230551908419 40.503735199315, -74.23077122522 40.504359026576))

In [ ]:
from shapely import wkt

geom = wkt.loads("MULTILINESTRING ((-74.003990902922 40.633997793231, -74.004550255018 40.633422475922))")

coords = []
for line in geom.geoms:
    coords.extend(list(line.coords))

In [ ]:
coords

In [ ]:
cscl.shape

In [ ]:
import pandas as pd
from datetime import datetime

def iter_diffs(csv_path):
    last_seen = {}  # key = (physical_id, day), value = last timestamp

    for chunk in pd.read_csv(csv_path, usecols=['physical_id', 'snapshot'], chunksize=500_000):
        for pid, snap in zip(chunk['physical_id'], chunk['snapshot']):
            try:
                t = datetime.strptime(snap, "%Y-%m-%dT%H:%M:%S.%f")
            except ValueError:
                continue
            day = t.date()
            key = (pid, day)
            if key in last_seen:
                diff_min = (t - last_seen[key]).total_seconds() / 60
                if diff_min < 13 or diff_min > 17:  # outside 13–17 min
                    print(f"physical_id: {pid}, day: {day}, prev: {last_seen[key]}, curr: {t}, diff_min: {diff_min:.2f}")
                yield diff_min
            last_seen[key] = t




In [ ]:
csv_files = [f"data/raw/plow/plow_{i}.csv" for i in {1}]  # all files
diffs = []

for filename in csv_files:
    print(f"Processing {filename}...")
    diffs.extend(iter_diffs(filename))

s = pd.Series(diffs)
s = s[s < 60]  # filter out gaps >= 1 hour

summary = {
    "avg_diff_min": s.mean(),
    "median_diff_min": s.median(),
    "std_diff_min": s.std(),
    "n_diffs": len(s),
    "prop_near_15": ((s >= 13) & (s <= 17)).mean()
}

print(summary)


In [ ]:
# get unique ids

import pandas as pd

unique_ids = set()

# Iterate over all files
for i in range(40):  # adjust to your number of CSVs
    filename = f"data/raw/plow/plow_{i}.csv"
    print(f"Processing {filename}...")

    # Read in chunks to handle large files
    for chunk in pd.read_csv(filename, usecols=['physical_id'], chunksize=500_000):
        unique_ids.update(chunk['physical_id'].unique())

print(f"Total unique physical_id: {len(unique_ids)}")


In [ ]:
# Unique PHYSICALID values from CSCL
cscl_ids = set(cscl['PHYSICALID'].unique())

# Find which CSCL IDs never appeared in the Plow dataset
never_plowed = cscl_ids - unique_ids

print(f"Number of streets never plowed: {len(never_plowed)}")
print("Streets never plowed:", [int(x) for x in list(never_plowed)])


In [ ]:
cscl[cscl["PHYSICALID"] == 68210]

In [ ]:
cscl[["Borough Code", "Full Street Name"]].head(1000)

In [ ]:
import pandas as pd

def compute_hourly_coverage(csv_path):
    # Read only necessary columns
    df = pd.read_csv(csv_path, usecols=['physical_id', 'last_visited', 'snapshot'])

    # Convert snapshot to datetime
    df['snapshot'] = pd.to_datetime(df['snapshot'], errors='coerce')
    df = df.dropna(subset=['snapshot'])

    # Floor snapshot to the hour
    df['snap_hour'] = df['snapshot'].dt.floor('H')

    # Group by physical_id and snap_hour
    grouped = df.groupby(['physical_id', 'snap_hour'], observed=True)

    # Compute unique last_visited / total snapshots
    result = grouped.agg(
        unique_visits=('last_visited', 'nunique'),
        n_snapshots=('snapshot', 'count')
    ).reset_index()

    # Compute coverage, handling divide-by-zero
    result['coverage'] = result.apply(
        lambda r: 0 if r['n_snapshots'] == 0 else r['unique_visits'] / r['n_snapshots'],
        axis=1
    )

    # Drop intermediate columns if not needed
    result = result[['physical_id', 'snap_hour', 'coverage']]
    return result


# Process all plow_i files individually
for i in range(10,40):
    path = f'data/raw/plow/plow_{i}.csv'
    print(f'Processing {path}...')
    hourly_cov = compute_hourly_coverage(path)
    hourly_cov.to_csv(f'data/derived/plow_coverage/plow_coverage_{i}.csv', index=False)


In [ ]:
for i in range(40):
    df = pd.read_csv(f"data/derived/plow_coverage/plow_coverage_{i}.csv")
    df["snap_hour"] = pd.to_datetime(df["snap_hour"])
    df_2020_winter = df[
    (df["snap_hour"].dt.year == 2020) &
    (df["snap_hour"].dt.month.isin([1, 2, 3]))
    ]
    print(df_2020_winter)

empty!

In [ ]:
for i in range(40):
    df = pd.read_csv(f"data/derived/plow_coverage/plow_coverage_{i}.csv")
    df["snap_hour"] = pd.to_datetime(df["snap_hour"])
    df_2019_winter = df[
    (df["snap_hour"].dt.year == 2019) &
    (df["snap_hour"].dt.month.isin([11, 12]))
    ]
    print(df_2019_winter)

also empty!

sanity check...

In [ ]:
for i in range(5):
    df = pd.read_csv(f"data/derived/plow_coverage/plow_coverage_{i}.csv")
    df["snap_hour"] = pd.to_datetime(df["snap_hour"])
    print(df.head(10))
    df_2018_winter = df[
    (df["snap_hour"].dt.year == 2018) &
    (df["snap_hour"].dt.month.isin([11, 12]))
    ]
    print(df_2018_winter)